In [1]:
from util import *

directory = r'../../data\output/'

%matplotlib notebook

[nltk_data] Downloading package punkt to C:\Users\WK/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# read data
df_documents = pd.read_csv(directory+'df_documents_sector.csv')
# df = pd.read_csv(directory+'df_documents_funda.csv')

Apply fine-tuned models - monetary

In [ ]:
# stance classification

# type_dict = {'staff': 'IMF staff', 'buff': 'country\'s authority'}
type_dict1 = {'staff': 'IMF staff', 'buff': 'a country\'s authorities'}
type_dict2 = {'staff': 'IMF staff\'s', 'buff': 'country\'s authorities\''}
verb_dict = {'staff': 'recommended', 'buff': 'planned'}
explanation_dict = {'staff': 'Note that the near-term policy direction recommended by staff is different in concept from staff\'s projected near-term policy direction of the authorities.',
                    'buff': 'Note that the near-term policy direction planned by the authorities are different in concept from IMF staff\'s recommended or projected near-term policy direction.'}

model_id = 'ft:gpt-4o-mini-2024-07-18:personal::A4qneai9'  # stance

for i,row in df_documents[(df_documents['sector']=='Monetary')&(df_documents['monetary_sample'])&(~df_documents['staff'].isna())&(~df_documents['buff'].isna())].iterrows():
    for ty in ['staff', 'buff']:
        if len(row[ty]) > 0:
            chat_completion = client.chat.completions.create(
                messages=[
                        {   "role": "system",
                            "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year expressing the views of %s, complete the following two tasks. First, classify the country\'s recent or current monetary policy stance as described in the text into "restrictive"/"neutral"/"accommodative"/"unclear"/"irrelevant"; if it discusses monetary policy but the specific stance is not clear, assign "unclear"; if it does not discuss monetary policy, assign "irrelevant". Second, classify the %s %s near-term (next year) direction of change in monetary policy stance as described in the text into "tightening"/"tightening bias"/"no change"/"loosening bias"/"loosening"/"unclear"/"irrelevant". %s If the text indicates a leaning towards a tightening/loosening monetary policy but without a full commitment, assign "tightening bias"/"loosening bias". If the text discusses monetary policy stance but the direction of change is not clear, assign "no change". If it does not discuss monetary policy stance but discusses monetary policy, assign "unclear". If it does not discuss monetary policy, assign "irrelevant". Return a JSON dict without additional texts: \"stance_current\", \"stance_future\".'''%(type_dict1[ty],  type_dict2[ty],  verb_dict[ty], explanation_dict[ty]),},
                        {   "role": "user",
                            "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row[ty])}
                ],
                model=model_id,
                temperature=0
            )
            try:
                result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```','').replace("\'", "\""))
                df_documents.loc[i,'stance_current_gpt_ft_%s'%ty] = result['stance_current']
                df_documents.loc[i,'stance_future_gpt_ft_%s'%ty] = result['stance_future']
            except Exception:
                df_documents.loc[i, 'error_gpt_ft_%s'%ty] = chat_completion.choices[0].message.content

# df_documents['stance_current_gpt_ft_staff'] = df_documents['stance_current_gpt_ft_staff'].apply(lambda x: x.replace('tightening bias', 'restrictive').replace('loose', 'accommodative'))
# df_documents['stance_future_gpt_ft_staff'] = df_documents['stance_future_gpt_ft_staff'].apply(lambda x: x.replace('restrictive', 'tightening'))

In [ ]:
df_documents['error_gpt_ft_%s'%ty].value_counts()

In [ ]:
model_id = 'ft:gpt-4o-mini-2024-07-18:personal::A4qCO7QC'  # agreement

for i,row in df_documents[(df_documents['sector']=='Monetary')&(df_documents['monetary_sample'])&(~df_documents['staff'].isna())&(~df_documents['buff'].isna())].iterrows():
    if len(row['staff'])>0 and len(row['buff'])>0:
        chat_completion = client.chat.completions.create(
            messages=[
                    {   "role": "system",
                        "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts expressing the views of IMF staff and a country's authorities, respectively, your task is to determine whether the authorities agree or disagree with IMF staff on issues related to the country's monetary policy. First, assign a value to the "agreement" field": "mostly agree"/"disagreement exists"/"irrelevant". Note that the authorities' agreement with IMF staff's views is different in concept from IMF staff's agreement with the authorities' views. If the two pieces of texts discuss common aspect(s) of monetary policy or if the authorities directly express agreement/disagreement to monetary related issues in either text: (a) if the authorities disagree with IMF staff on any monetary policy issues, assign "disagreement exists"; (b) if there is no disagreement by the authorities, assign "mostly agree". If the authorities do not directly express agreement/disagreement with IMF staff on monetary related issues, and either of the texts does not discuss monetary policy or they discuss entirely different aspects of monetary policy, (c) assign "irrelevant".  Second, if disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; for example, "current policy stance", "future policy direction", "monetary policy framework", "monetary policy operations", "central bank communication", "institutions", "economic assessment", etc; if the authorities do not disagree with staff, leave the "disagreement_areas" field blank. Return a JSON dict without additional texts: "agreement", "disagreement_areas".''',},
                    {   "role": "user",
                        "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])},
            ],
            model=model_id,
            temperature=0
        )

        try:
            result = json.loads(chat_completion.choices[0].message.content.replace("'", '"').replace('```json','').replace('```','').replace('\n',' '))
            df_documents.loc[i,'agreement_gpt_ft'] = result['agreement']
            df_documents.loc[i,'disagreement_areas_gpt_ft'] = result['disagreement_areas']
        except Exception:
            df_documents.loc[i, 'error_gpt_ft'] = chat_completion.choices[0].message.content

df_documents['error_gpt_ft'] = df_documents['error_gpt_ft'].apply(lambda x: json.loads(x.replace("'", '"').replace('s" ', "s' ").replace('"s ', "'s ").replace('```json','').replace('```','').replace('\n',' ').strip('.')) if x==x and x not in ['nan', 'n'] else np.nan)
df_documents['agreement_gpt_ft'] = df_documents.apply(lambda x: x['error_gpt_ft']['agreement'] if x['error_gpt_ft']==x['error_gpt_ft'] and (x['agreement_gpt_ft']!=x['agreement_gpt_ft'] or x['agreement_gpt_ft'] in ['nan', 'n']) else x['agreement_gpt_ft'], axis=1)
df_documents['disagreement_areas_gpt_ft'] = df_documents.apply(lambda x: x['error_gpt_ft']['disagreement_areas'] if x['error_gpt_ft']==x['error_gpt_ft'] and (x['disagreement_areas_gpt_ft']!=x['disagreement_areas_gpt_ft'] or x['disagreement_areas_gpt_ft'] in ['nan', 'n']) else x['disagreement_areas_gpt_ft'], axis=1)


In [ ]:
df_documents['disagreement_areas_gpt_ft'] = df_documents['disagreement_areas_gpt_ft'].apply(lambda x: '; '.join(x) if x==x and x!='nan' else np.nan)

In [ ]:
# df_documents.to_csv(directory+'df_documents_sector_labelled.csv', index=False)
df_documents[(df_documents['sector']=='Monetary')&(df_documents['monetary_sample'])&(~df_documents['staff'].isna())&(~df_documents['buff'].isna())].to_csv(directory+'gpt/df_documents_sector_labelled_mon_v2.csv', index=False)

Apply fine-tuned models - fiscal

In [ ]:
# df_documents = pd.read_csv(directory+'df_documents_sector_labelled.csv')

# stance classification
type_dict = {'staff': 'IMF staff', 'buff': 'country\'s authority'}
model_id = 'ft:gpt-4o-mini-2024-07-18:personal::A1cjIYxS' # stance

for i,row in df_documents[(df_documents['sector']=='Fiscal')&(~df_documents['staff'].isna())&(~df_documents['buff'].isna())].iterrows():
    for ty in ['staff', 'buff']:
        if len(row[ty]) > 0:
            chat_completion = client.chat.completions.create(
                messages=[
                        {   "role": "system",
                            "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year expressing the views of %s, classify the %s\'s recommended or planned near-term (next year) direction of change in fiscal policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant; if it discusses fiscal policy but the direction of change is not clear, assign unclear; if it does not discuss fiscal policy, assign irrelevant. Return your answer without additional texts.'''%(type_dict[ty],  type_dict[ty]),},
                        {   "role": "user",
                            "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row[ty])}
                ],
                model=model_id,
            )
            result = chat_completion.choices[0].message.content
            df_documents.loc[i,'stance_near_term_gpt_ft_%s'%ty] = result

In [ ]:
df_documents['stance_near_term_gpt_ft_staff'] = df_documents['stance_near_term_gpt_ft_staff'].apply(lambda x: 'loosening bias' if x in ['loose bias', 'loosening and bias'] else x)
df_documents['stance_near_term_gpt_ft_buff'] = df_documents['stance_near_term_gpt_ft_buff'].apply(lambda x: x.replace('........................................................................................... \ntightening', 'tightening'))

In [22]:
model_id = 'ft:gpt-4o-mini-2024-07-18:personal::A4q8cvri'  # agreement

for i,row in df_documents[(df_documents['sector']=='Fiscal')&(~df_documents['staff'].isna())&(~df_documents['buff'].isna())].iterrows():
    if len(row['staff'])>0 and len(row['buff'])>0:
        chat_completion = client.chat.completions.create(
            messages=[
                    {   "role": "system",
                        "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts expressing the views of IMF staff and a country's authorities, respectively, your task is to determine whether the authorities agree or disagree with IMF staff on issues related to the country's fiscal policy. First, assign a value to the "agreement" field": "mostly agree"/"disagreement exists"/"irrelevant". Note that the authorities' agreement with IMF staff's views is different in concept from IMF staff's agreement with the authorities' views. If the two pieces of texts discuss common aspect(s) of fiscal policy or if the authorities directly express agreement/disagreement to fiscal related issues in either text: (a) if the authorities disagree with IMF staff on any fiscal policy issues, assign "disagreement exists"; (b) if there is no disagreement by the authorities, assign "mostly agree". If the authorities do not directly express agreement/disagreement with IMF staff on fiscal related issues, and either of the texts does not discuss fiscal policy or they discuss entirely different aspects of fiscal policy, (c) assign "irrelevant".  Second, if disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; for example, "near-term policy direction", "fiscal revenue", "fiscal expenditure", "government debt & financing", "economic assessment", "fiscal framework", "medium-term fiscal stance", etc; if the authorities do not disagree with staff, leave the "disagreement_areas" field blank. Return a JSON dict without additional texts: "agreement", "disagreement_areas".''',},
                    {   "role": "user",
                        "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])},
            ],
            model=model_id,
            temperature=0
        )

        try:
            result = json.loads(chat_completion.choices[0].message.content.replace("'", '"').replace('```json','').replace('```','').replace('\n',' '))
            df_documents.loc[i,'agreement_gpt_ft'] = result['agreement']
            df_documents.loc[i,'disagreement_areas_gpt_ft'] = result['disagreement_areas']
        except Exception:
            df_documents.loc[i, 'error_gpt_ft'] = chat_completion.choices[0].message.content

df_documents['error_gpt_ft'] = df_documents['error_gpt_ft'].apply(lambda x: json.loads(x.replace("'", '"').replace('s" ', "s' ").replace('"s ', "'s ").replace('```json','').replace('```','').replace('\n',' ').strip('.').replace('")', '"]')) if x==x and x not in ['nan', 'n'] else np.nan)
for i, row in df_documents.iterrows():
    if type(row['error_gpt_ft'])==dict and 'agree' in row['error_gpt_ft']:
        row['error_gpt_ft']['agreement'] = row['error_gpt_ft']['agree']
        df_documents.at[i, 'error_gpt_ft'] = row['error_gpt_ft']

df_documents['agreement_gpt_ft'] = df_documents.apply(lambda x: x['error_gpt_ft']['agreement'] if x['error_gpt_ft']==x['error_gpt_ft'] and (x['agreement_gpt_ft']!=x['agreement_gpt_ft'] or x['agreement_gpt_ft'] in ['nan', 'n']) else x['agreement_gpt_ft'], axis=1)
df_documents['disagreement_areas_gpt_ft'] = df_documents.apply(lambda x: x['error_gpt_ft']['disagreement_areas'] if x['error_gpt_ft']==x['error_gpt_ft'] and (x['disagreement_areas_gpt_ft']!=x['disagreement_areas_gpt_ft'] or x['disagreement_areas_gpt_ft'] in ['nan', 'n']) else x['disagreement_areas_gpt_ft'], axis=1)


In [23]:
df_documents[df_documents['sector']=='Fiscal']['disagreement_areas_gpt_ft'].value_counts()

,count
disagreement_areas_gpt_ft,
[],474
[fiscal revenue],133
[government debt & financing],90
[fiscal expenditure],87
[economic assessment],85
[fiscal framework],30
[medium-term fiscal stance],30
[near-term policy direction],20
"[government debt & financing, near-term policy direction]",9


In [24]:
df_documents['agreement_gpt_ft'].value_counts()

,count
agreement_gpt_ft,
disagreement exists,502
mostly agree,422
irrelevant,52


In [ ]:
stance_dict = {'tightening': 5, 'tightening bias': 4, 'no change': 3, 'loosening bias': 2, 'loosening': 1}
df_documents['stance_near_term_gpt_ft_staff_num'] = df_documents['stance_near_term_gpt_ft_staff'].apply(lambda x: stance_dict[x] if x in stance_dict else np.nan)
df_documents['stance_near_term_gpt_ft_buff_num'] = df_documents['stance_near_term_gpt_ft_buff'].apply(lambda x: stance_dict[x] if x in stance_dict else np.nan)
df_documents['agreement_stance_near_term_gpt_ft_num'] = df_documents['stance_near_term_gpt_ft_staff_num']-df_documents['stance_near_term_gpt_ft_buff_num']
df_documents['agreement_stance_near_term_gpt_ft_cate1'] = df_documents['agreement_stance_near_term_gpt_ft_num'].apply(lambda x: 'major difference' if x >= 3 or x <= -3 else 'some difference' if x in [2, -2] else 'minor difference' if x in [1, -1] else 'no difference' if x==0 else 'irrelevant')
df_documents['agreement_stance_near_term_gpt_ft_cate2'] = df_documents['agreement_stance_near_term_gpt_ft_num'].apply(lambda x: 'significantly tighter' if x >= 3 else 'tighter' if x==2 else 'moderately tighter' if x==1 else 'same' if x==0 else 'moderately looser' if x==-1 else 'looser' if x==-2 else 'significantly looser' if x<=-3 else 'irrelevant')

In [25]:
# save
# df_documents.to_csv(directory+'df_documents_sector_labelled.csv', index=False)
df_documents[(df_documents['sector']=='Fiscal')&(~df_documents['staff'].isna())&(~df_documents['buff'].isna())].to_csv(directory+'gpt/df_documents_sector_labelled_fis_v2.csv', index=False)

fiscal stance revisit

In [5]:
df_documents = pd.read_csv(directory+'df_documents_sector.csv')
df_documents = df_documents[(df_documents['sector']=='Fiscal')&(~df_documents['staff'].isna())&(~df_documents['buff'].isna())]
df_documents['staff_sa'] = df_documents.apply(lambda x: '\n'.join([p for p in x['staff'].split('\n') if p in x['paragraphs_sa']]).strip(), axis=1)

type_dict1 = {'staff': 'IMF staff', 'buff': 'a country\'s authorities'}
type_dict2 = {'staff': 'IMF staff\'s', 'buff': 'country\'s authorities\''}
verb_dict = {'staff': 'recommended', 'buff': 'planned'}
explanation_dict = {'staff': 'Note that the near-term policy direction recommended by staff is different in concept from staff\'s projected near-term policy direction of the authorities.',
                    'buff': 'Note that the near-term policy direction planned by the authorities are different in concept from IMF staff\'s recommended or projected near-term policy direction.'}

model_id = 'ft:gpt-4o-mini-2024-07-18:personal::A4ZD00Na' # stance

for i,row in df_documents.iterrows():
    for ty in ['staff']:
        if len(row[ty]) > 0:
            chat_completion = client.chat.completions.create(
                messages=[
                        {   "role": "system",
                            "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year expressing the views of %s, classify the %s %s near-term (next year) direction of change in fiscal policy stance as described in the text into "tightening"/"tightening bias"/"no change"/"loosening bias"/"loosening"/"unclear"/"irrelevant". %s If the text indicates a leaning towards a tightening/loosening fiscal policy but without a full commitment, assign "tightening bias"/"loosening bias". If the text discusses fiscal policy but the direction of change is not clear, assign "unclear"; if it does not discuss fiscal policy, assign "irrelevant". Return your answer without additional texts.'''%(type_dict1[ty],  type_dict2[ty],  verb_dict[ty], explanation_dict[ty]),},
                        {   "role": "user",
                            "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row[ty+'_sa'])}
                ],
                model=model_id,
                temperature=0
            )
            result = chat_completion.choices[0].message.content
            df_documents.loc[i,'stance_near_term_gpt_ft_%s'%ty] = result
    for ty in ['buff']:
        if len(row[ty]) > 0:
            chat_completion = client.chat.completions.create(
                messages=[
                        {   "role": "system",
                            "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year expressing the views of %s, classify the %s %s near-term (next year) direction of change in fiscal policy stance as described in the text into "tightening"/"tightening bias"/"no change"/"loosening bias"/"loosening"/"unclear"/"irrelevant". %s If the text indicates a leaning towards a tightening/loosening fiscal policy but without a full commitment, assign "tightening bias"/"loosening bias". If the text discusses fiscal policy but the direction of change is not clear, assign "unclear"; if it does not discuss fiscal policy, assign "irrelevant". Return your answer without additional texts.'''%(type_dict1[ty],  type_dict2[ty],  verb_dict[ty], explanation_dict[ty]),},
                        {   "role": "user",
                            "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row[ty])}
                ],
                model=model_id,
                temperature=0
            )
            result = chat_completion.choices[0].message.content
            df_documents.loc[i,'stance_near_term_gpt_ft_%s'%ty] = result

In [7]:
df_documents['stance_near_term_gpt_ft_staff'].value_counts()

,count
stance_near_term_gpt_ft_staff,
tightening,602
loosening,144
unclear,110
no change,52
loosening bias,42
tightening bias,26


In [64]:
df_documents['stance_near_term_gpt_ft_staff'] = df_documents['stance_near_term_gpt_ft_staff'].apply(lambda x: 'unclear' if 'unclear' in x.lower() else 'tightening bias' if 'tight' in x and 'bias' in x.lower() else 'tightening' if 'tight' in x.lower() else 'loosening bias' if 'loos' in x.lower() and 'bias' in x.lower() else 'loosening' if 'loos' in x.lower() else 'no change' if 'no change' in x.lower() else x)
df_documents['stance_near_term_gpt_ft_staff'] = df_documents['stance_near_term_gpt_ft_staff'].apply(lambda x: 'unclear' if x not in ['tightening', 'loosening', 'unclear', 'no change', 'loosening bias',
       'tightening bias', 'irrelevant'] else x)

In [24]:
df_documents['stance_near_term_gpt_ft_staff'] = df_documents['stance_near_term_gpt_ft_staff'].apply(lambda x: x.replace("looseningIt's", 'loosening').replace("tighteningæl", 'tightening').replace('no change pappro', 'no change'))
df_documents['stance_near_term_gpt_ft_staff'] = df_documents['stance_near_term_gpt_ft_staff'].apply(lambda x: 'unclear' if x not in ['tightening', 'loosening', 'unclear', 'no change', 'loosening bias',
       'tightening bias', 'irrelevant'] else x)
df_documents['stance_near_term_gpt_ft_buff'] = df_documents['stance_near_term_gpt_ft_buff'].apply(lambda x: x.replace("With virtually no change", 'no change').replace("loosening ethnicity", 'loosening bias').replace('tightening df', 'tightening').replace('tightening smokers', 'tightening').replace('biased', 'unclear').strip())


In [14]:
# retry unclear or irrelevant obs
for i,row in df_documents.iterrows():
    if row['stance_near_term_gpt_ft_staff'] in ['unclear', 'irrelevant']:
        for ty in ['staff']:
            if len(row[ty]) > 0:
                chat_completion = client.chat.completions.create(
                    messages=[
                            {   "role": "system",
                                "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year expressing the views of %s, classify the %s %s near-term (next year) direction of change in fiscal policy stance as described in the text into "tightening"/"tightening bias"/"no change"/"loosening bias"/"loosening"/"unclear"/"irrelevant". %s If the text indicates a leaning towards a tightening/loosening fiscal policy but without a full commitment, assign "tightening bias"/"loosening bias". If the text discusses fiscal policy but the direction of change is not clear, assign "unclear"; if it does not discuss fiscal policy, assign "irrelevant". Return your answer without additional texts.'''%(type_dict1[ty],  type_dict2[ty],  verb_dict[ty], explanation_dict[ty]),},
                            {   "role": "user",
                                "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row[ty])}
                    ],
                    model=model_id,
                    temperature=0
                )
                result = chat_completion.choices[0].message.content
                df_documents.loc[i,'stance_near_term_gpt_ft_%s'%ty] = result
    if row['stance_near_term_gpt_ft_buff'] in ['unclear', 'irrelevant']:
        for ty in ['buff']:
            if len(row[ty]) > 0:
                chat_completion = client.chat.completions.create(
                    messages=[
                            {   "role": "system",
                                "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year expressing the views of %s, classify the %s %s near-term (next year) direction of change in fiscal policy stance as described in the text into "tightening"/"tightening bias"/"no change"/"loosening bias"/"loosening"/"unclear"/"irrelevant". %s If the text indicates a leaning towards a tightening/loosening fiscal policy but without a full commitment, assign "tightening bias"/"loosening bias". If the text discusses fiscal policy but the direction of change is not clear, assign "unclear"; if it does not discuss fiscal policy, assign "irrelevant". Return your answer without additional texts.'''%(type_dict1[ty],  type_dict2[ty],  verb_dict[ty], explanation_dict[ty]),},
                            {   "role": "user",
                                "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row[ty])}
                    ],
                    model=model_id,
                    temperature=0
                )
                result = chat_completion.choices[0].message.content
                df_documents.loc[i,'stance_near_term_gpt_ft_%s'%ty] = result

In [16]:
df_documents['stance_near_term_gpt_ft_staff'].value_counts()

,count
stance_near_term_gpt_ft_staff,
tightening,647
loosening,184
no change,57
loosening bias,44
tightening bias,30
unclear,14


In [ ]:
stance_dict = {'tightening': 5, 'tightening bias': 4, 'no change': 3, 'loosening bias': 2, 'loosening': 1}
df_documents['stance_near_term_gpt_ft_staff_num'] = df_documents['stance_near_term_gpt_ft_staff'].apply(lambda x: stance_dict[x] if x in stance_dict else np.nan)
df_documents['stance_near_term_gpt_ft_buff_num'] = df_documents['stance_near_term_gpt_ft_buff'].apply(lambda x: stance_dict[x] if x in stance_dict else np.nan)
df_documents['agreement_stance_near_term_gpt_ft_num'] = df_documents['stance_near_term_gpt_ft_staff_num']-df_documents['stance_near_term_gpt_ft_buff_num']
df_documents['agreement_stance_near_term_gpt_ft_cate1'] = df_documents['agreement_stance_near_term_gpt_ft_num'].apply(lambda x: 'major difference' if x >= 3 or x <= -3 else 'some difference' if x in [2, -2] else 'minor difference' if x in [1, -1] else 'no difference' if x==0 else 'irrelevant')
df_documents['agreement_stance_near_term_gpt_ft_cate2'] = df_documents['agreement_stance_near_term_gpt_ft_num'].apply(lambda x: 'significantly tighter' if x >= 3 else 'tighter' if x==2 else 'moderately tighter' if x==1 else 'same' if x==0 else 'moderately looser' if x==-1 else 'looser' if x==-2 else 'significantly looser' if x<=-3 else 'irrelevant')

In [19]:
# df_documents.to_excel(directory+'gpt/df_documents_sector_fis_revisited_ts.xlsx', index=False, engine='xlsxwriter')
df_documents.to_csv(directory+'gpt/df_documents_sector_fis_revisited_ts.csv', index=False)